# Predicting the sale price of bulldozers using machine learning

In this notebook, we're going to go through an example machine learning project with the goal of predicting the sale priceo of bulldozers.

## 1. Problem definition
> How well can we predict the future sale priceo fa bulldozer, given the characteristic and previous examples of how much similar bulldozers have been sold for?
## 2. Data
Data is downlaoded from kaggle bluebook for bulldozers competition: 
https://www.kaggle.com/c/bluebook-for-bulldozers/data

There are 3 main datasets:
* Train.csv is the training set, which contains data through the end of 2011.
* Valid.csv is the validation set, which contains data from January 1, 2012 - April 30, 2012 You make predictions on this set throughout the majority of the competition. Your score on this set is used to create the public leaderboard.
* Test.csv is the test set, which won't be released until the last week of the competition. It contains data from May 1, 2012 - November 2012. Your score on the test set determines your final rank for the competition.
## 3. Evaludation
The evaluation metric for this competition is the RMSLE (root mean squared log error) between the actual and predicted auction prices.
For more on the evaluation of this project check: https://www.kaggle.com/c/bluebook-for-bulldozers/overview/evaluation

Note: The goal for most regression evaluation metrics is to minimize the error. For example, our goal for this project will be ot build a machine learning model that minimizes RMSLE.
## 4. Features
Kaggle provides a data dictionary detailing all of the features of the dataset. You can view this data dictionary on Google sheets:
https://docs.google.com/spreadsheets/d/18ly-bLR8sbDJLITkWG7ozKm8l3RyieQ2Fpgix-beSYI/edit?usp=sharing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

In [ ]:
# import training and validatio nsets
df = pd.read_csv('data/trainandvalid.csv', low_memory=False)
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))
ax.scatter(df['saledate'][:1000], df['SalePrice'][:1000])


In [ ]:
df.SalePrice.plot.hist(color="lightblue")

### Parsing dates
When we work with time series data, we want to enrich the time & date component as much as possible.

We can do that by telling pandas which of our columns has dates in it using the `parse_dates` parameter.

In [ ]:
# Import data again but this time parse dates
df = pd.read_csv(
    'data/trainandvalid.csv', 
    low_memory=False, 
    parse_dates=['saledate'])
df.head()

In [ ]:
df.saledate.dtype

In [ ]:
df.saledate[:1000]

In [ ]:
fig, ax = plt.subplots()
ax.scatter(df['saledate'][:1000], df['SalePrice'][:1000])

In [ ]:
df.head().T

In [ ]:
df.saledate.head(20)

### Sort Dataframe by saledate
When working with time series data, it's a good idea to sort it by date.

In [ ]:
# Sort Dataframe in date order
df.sort_values(by='saledate', inplace=True, ascending=True)
df.saledate.head(20)

In [ ]:
df.head()

### Make a copy of the original Dataframe
We make a copy of the original dataframe so when we manipulate the copy, we've still got our original data.

In [ ]:
# Make a copy
df_tmp = df.copy()
df_tmp.saledate

### Add datetime parameteres for `saledate` column

In [ ]:
df_tmp["saleYear"] = df_tmp['saledate'].dt.year
df_tmp["saleMonth"] = df_tmp['saledate'].dt.month
df_tmp["saleDay"] = df_tmp['saledate'].dt.day
df_tmp["saleDayOfWeek"] = df_tmp['saledate'].dt.dayofweek
df_tmp["saleDayOfYear"] = df_tmp['saledate'].dt.dayofyear

In [ ]:
df_tmp.head().T

In [ ]:
df_tmp.drop("saledate", axis=1, inplace=True)

In [ ]:
# Check the value of different columns
df_tmp.state.value_counts()


## 5. Modeling
We've done enough EDA (we could always do more) but let's start to do some model-driven EDA.

In [ ]:
df_tmp.info()

In [ ]:
df_tmp["UsageBand"].dtype

In [ ]:
df_tmp.isna().sum()

### Convert string to categories
ONe way we can turn all of our data into numbers is by converting them into pandas categories.

We can check the different datatypes compatible with pandas here: https://pandas.pydata.org/pandas-docs/stable/reference/general_utility_functions.html#data-types-related-functionality

In [ ]:
pd.api.types.is_string_dtype(df_tmp.UsageBand)

In [ ]:
# Find the columns which contain strings
for label, columns in df_tmp.items():
    if pd.api.types.is_string_dtype(columns):
        print(label)

In [ ]:
# This will turn all of the string valyes in to category values
for label, columns in df_tmp.items():
    if pd.api.types.is_string_dtype(columns):
        df_tmp[label] = df_tmp[label].astype('category').cat.as_ordered()

In [ ]:
df_tmp.info()

In [ ]:
df_tmp.state.cat.categories

In [ ]:
df_tmp.state.cat.codes

Thanks to pandas categories, we now have a way to access all of our data in the form of numbers.

But we still have a bunch of missing data


In [ ]:
# CHeck missing data
df_tmp.isna().sum()/len(df_tmp)

In [ ]:
# Export current tmp dataframe
df_tmp.to_csv("data/train_tmp.csv", index=False)

In [ ]:
# Import preprocessed data
df_tmp = pd.read_csv("data/train_tmp.csv", low_memory=False)
df_tmp.head().T

## Fill missing values
### Fill numeric missing values

In [ ]:

for label, columns in df_tmp.items():
    if pd.api.types.is_numeric_dtype(columns):
        print(label)


In [ ]:
df_tmp.ModelID

In [ ]:
for label, content in df_tmp.items():
    if pd.api.types.is_numeric_dtype(content):
        if pd.isnull(content).sum():
            print(label)

In [ ]:
# Fill numeric rows with the median
for label, content in df_tmp.items():
    if pd.api.types.is_numeric_dtype(content):
        if pd.isnull(content).sum():
            # ADd a binary column which tells us if the data was missing or not
            df_tmp[label + "_is_missing"] = pd.isnull(content)
            # Fill in the missing data
            df_tmp[label] = content.fillna(content.median())

In [ ]:
len(df_tmp)

In [ ]:
for label, content in df_tmp.items():
    if pd.api.types.is_numeric_dtype(content):
        if pd.isnull(content).sum():
            print(label)

In [ ]:
#check to see how many examples were  missing
df_tmp.auctioneerID_is_missing.value_counts()

### Filling and turning categorical variables into numbers

In [ ]:
# Check for columns which aren't numeric
for label, content in df_tmp.items():
    if not pd.api.types.is_numeric_dtype(content):
        print(label)

In [ ]:
# Turns categorical variables into numbers and fill missing
for label, content in df_tmp.items():
    if not pd.api.types.is_numeric_dtype(content):
        # Add binary column which tells us if the data was missing or not
        df_tmp[label + "_is_missing"] = pd.isnull(content)
        # Turn categories into numbers and add +1
        df_tmp[label] = pd.Categorical(content).codes + 1

In [ ]:
pd.Categorical(df_tmp.UsageBand).codes

In [ ]:
df_tmp.head().T

In [ ]:
df_tmp.isna().sum()

Now that all of our data in numeric as well as our dataframe has no missing values, we should be able to build a machine learning model.

In [ ]:
len(df_tmp)

In [ ]:
%%time
from sklearn.ensemble import RandomForestRegressor


# Instantiale model
model = RandomForestRegressor(n_jobs=-1, random_state=42)

# Fit the model
model.fit(df_tmp.drop("SalePrice", axis=1), df_tmp.SalePrice) 

In [ ]:
# Score model
model.score(df_tmp.drop("SalePrice", axis=1), df_tmp.SalePrice)